# 042 — Convert selected records to JSON

Reads the combined ESM and NGA-Sub conversion lists from `040` and writes one
per-record JSON file per selected `(record_identifier, component)` into
`cfg["proc_data"]["gm_records"]`, giving both databases a single, uniform
on-disk schema for the downstream MSA runs (`050`). Records are pulled from the
(network-mounted) raw archives in parallel; rows whose JSON already exists are
skipped, so the notebook is cheap and safe to re-run as more records arrive.

The conversion logic lives in two importable command-line scripts in
`phd_project/scripts/data_handling/` — `convert_esm_records_to_json.py`
(ASDF/HDF5 → JSON) and `convert_ngasub_records_to_json.py` (AT2 → JSON). Here we
call their `convert_*_records()` run-functions; see **Command-line usage** below
to run them standalone.

**Upstream** (from `040`, in `cfg["proc_data"]["gm_selection"]`):
- `esm_records_to_convert.csv`, `ngasub_records_to_convert.csv` — combined
  `(record_identifier, component)` conversion lists across both AvgSA campaigns.

Also reads the raw record archives on the share
(`cfg["raw_data"]["esm_hdf5_folder"]`, `cfg["raw_data"]["ngasub_folder"]`) and,
for NGA-Sub, the reference flatfiles `NGASub_SA_H1_H2_filenamemap.csv`
(H1/H2 → physical channel) and `NGASub_Metadata_SA_rotD50.csv`
(event/station codes) in `cfg["raw_data"]["gm_flatfiles"]`.

**Output** (in `cfg["proc_data"]["gm_records"]`):
- One `{record_identifier}__{component}.json` per converted record, sharing a
  common schema (`eq_index`, `database`, `dt`, `duration`, `units`,
  `record_type`, `component`, event/station codes, and the acceleration
  `record` in g, last).

**Run order** — run top to bottom after `040` has (re)written the conversion
lists. Records still missing from the raw archives are reported as skipped for
inspection rather than aborting the run.

In [1]:
from pathlib import Path

import pandas as pd

from phd_project.config import config
from phd_project.scripts.data_handling.convert_esm_records_to_json import (
    convert_esm_records,
)
from phd_project.scripts.data_handling.convert_ngasub_records_to_json import (
    convert_ngasub_records,
)

cfg = config.load_config()

## Paths

In [2]:
# Source record archives (network share - external resource)
ESM_SRC = cfg["raw_data"]["esm_hdf5_folder"]
NGASUB_SRC = cfg["raw_data"]["ngasub_folder"]

# Combined selection lists (from wp1pt3pt8i)
esm_selection_csv = cfg["proc_data"]["gm_selection"] / "esm_records_to_convert.csv"
ngasub_selection_csv = cfg["proc_data"]["gm_selection"] / "ngasub_records_to_convert.csv"

# NGA-Sub reference flatfiles
ngasub_filename_map = cfg["raw_data"]["gm_flatfiles"] / "NGASub_SA_H1_H2_filenamemap.csv"
ngasub_metadata = cfg["raw_data"]["gm_flatfiles"] / "NGASub_Metadata_SA_rotD50.csv"

# Output folder for the converted JSON records
output_dir = cfg["proc_data"]["gm_records"]

## Convert ESM records

In [3]:
n_esm, esm_skipped = convert_esm_records(
    ESM_SRC, esm_selection_csv, output_dir)

Converting ESM records: 0it [00:00, ?it/s]


Done: 0 converted, 942 already present (skipped), 0 failed (max_workers=125).


## Convert NGA-Sub records

In [4]:
n_ngasub, ngasub_skipped = convert_ngasub_records(
    NGASUB_SRC, ngasub_selection_csv, output_dir,
    filename_map=ngasub_filename_map, metadata=ngasub_metadata)

Converting NGA-Sub records:   0%|          | 0/554 [00:00<?, ?it/s]


Done: 499 converted, 557 already present (skipped), 55 failed (max_workers=125).
Skipped records:
  7001542 H1: AT2 file not found
  7006351 H1: AT2 file not found
  7008189 H2: AT2 file not found
  7002763 H2: AT2 file not found
  7003468 H2: AT2 file not found
  7002931 H2: AT2 file not found
  7007019 H1: AT2 file not found
  7002316 H2: AT2 file not found
  7005115 H2: AT2 file not found
  7001174 H1: AT2 file not found
  7007901 H1: AT2 file not found
  7004994 H2: AT2 file not found
  6003837 H1: AT2 file not found
  7002029 H2: AT2 file not found
  4033673 H2: AT2 file not found
  7004921 H2: AT2 file not found
  4033865 H1: AT2 file not found
  4038119 H2: AT2 file not found
  4018077 H2: AT2 file not found
  4012459 H1: AT2 file not found
  7008085 H2: AT2 file not found
  4033732 H2: AT2 file not found
  7008146 H1: AT2 file not found
  7008060 H2: AT2 file not found
  7001976 H1: AT2 file not found
  7001268 H2: AT2 file not found
  7004906 H2: AT2 file not found
  4009391 

## Records that could not be converted

Any selected records that were skipped (e.g. missing source file) are listed
below for inspection.

In [ ]:
esm_skipped_df = pd.DataFrame(esm_skipped, columns=["record_identifier", "component", "reason"])
ngasub_skipped_df = pd.DataFrame(ngasub_skipped, columns=["record_identifier", "component", "reason"])

print(f"ESM skipped: {len(esm_skipped_df)} | NGASub skipped: {len(ngasub_skipped_df)}")
display(esm_skipped_df)
display(ngasub_skipped_df)

ESM skipped: 0 | NGASub skipped: 633


,record_identifier,component,reason


,record_identifier,component,reason
0,7001179,H2,AT2 file not found
1,4032428,H2,AT2 file not found
2,4006380,H1,AT2 file not found
3,4018859,H2,AT2 file not found
4,4032414,H2,AT2 file not found
...,...,...,...
628,4022520,H2,AT2 file not found
629,4028069,H1,AT2 file not found
630,7006064,H1,AT2 file not found
631,4022463,H1,AT2 file not found
